# Welcome to notebooks!

Solidworks_Hackathon

upload solid_ai_hackathon.zip 

In [ ]:
import zipfile
import os


zip_path = 'solid_ai_hackathon.zip'

extract_dir = 'solid_ai_dataset'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f" Successfully extracted to: {extract_dir}")
print("Top-level contents after extraction:")
print(os.listdir(extract_dir))

Install libraries

In [ ]:
%uv pip install -q ultralytics huggingface_hub albumentations --upgrade

In [ ]:
import albumentations as A
import cv2
import numpy as np
from tqdm import tqdm
import shutil

def add_gaussian_noise_to_training_set(input_dir, output_dir):
    
    os.makedirs(output_dir, exist_ok=True)
    
    
    transform = A.Compose([
        A.GaussNoise(var_limit=(10.0, 50.0), mean=0, p=1.0),
    ])
    
    image_files = [f for f in os.listdir(input_dir) if f.endswith('.png')]
    
    print(f" Applying Gaussian Noise to {len(image_files)} images...")
    
    for img_name in tqdm(image_files, desc="Adding noise"):
        img_path = os.path.join(input_dir, img_name)
        img = cv2.imread(img_path)
        
        if img is None:
            continue
            
      
        augmented = transform(image=img)
        noisy_img = augmented['image']
        
       
        output_path = os.path.join(output_dir, img_name)
        cv2.imwrite(output_path, noisy_img)
    
    print(f" Noisy images saved to: {output_dir}")
    return output_dir


NOISY_TRAIN_DIR = add_gaussian_noise_to_training_set(
    input_dir=TRAIN_IMG_DIR,
    output_dir='solid_ai_dataset/Solid_Ai_Hackathon/train/train_noisy'
)

In [ ]:
import pandas as pd
import os
import shutil
import cv2
from sklearn.model_selection import train_test_split
from tqdm import tqdm


BASE_DIR = 'solid_ai_dataset/Solid_Ai_Hackathon'
CSV_PATH = os.path.join(BASE_DIR, 'train_bboxes.csv')
TRAIN_IMG_DIR = 'solid_ai_dataset/Solid_Ai_Hackathon/train/train_noisy'  # CHANGED: Use noisy images
OUTPUT_DIR = 'yolo_dataset'


CLASS_MAP = {'bolt': 0, 'locatingpin': 1, 'nut': 2, 'washer': 3}


def convert_to_yolo_bbox(row, w, h):
    x_center = (row['x_min'] + row['x_max']) / 2 / w
    y_center = (row['y_min'] + row['y_max']) / 2 / h
    width = (row['x_max'] - row['x_min']) / w
    height = (row['y_max'] - row['y_min']) / h
    return [x_center, y_center, width, height]


df = pd.read_csv(CSV_PATH)
unique_images = df['image_name'].unique()


train_imgs, val_imgs = train_test_split(unique_images, test_size=0.15, random_state=42)


for split in ['train', 'val']:
    os.makedirs(f'{OUTPUT_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUTPUT_DIR}/labels/{split}', exist_ok=True)


for img_name in tqdm(unique_images, desc="Converting to YOLO format"):
    split = 'train' if img_name in train_imgs else 'val'
    
    
    src = os.path.join(TRAIN_IMG_DIR, img_name)
    dst = os.path.join(OUTPUT_DIR, 'images', split, img_name)
    shutil.copy(src, dst)
    
    
    img = cv2.imread(src)
    h, w = img.shape[:2]
    
    
    img_bboxes = df[df['image_name'] == img_name]
    
    
    label_path = os.path.join(OUTPUT_DIR, 'labels', split, img_name.replace('.png', '.txt'))
    with open(label_path, 'w') as f:
        for _, row in img_bboxes.iterrows():
            cls_id = CLASS_MAP[row['class']]
            bbox = convert_to_yolo_bbox(row, w, h)
            f.write(f"{cls_id} {' '.join(map(str, bbox))}\n")

print(" YOLO dataset prepared with Gaussian noise augmentation!")
print(f"   Output folder: {OUTPUT_DIR}")
print(f"   Train images: {len(train_imgs)} | Validation images: {len(val_imgs)}")

In [ ]:
import yaml
import os


data_yaml = {
    'path': '',              
    'train': 'images/train',
    'val': 'images/val',
    'nc': 4,
    'names': ['bolt', 'locatingpin', 'nut', 'washer']
}


yaml_path = 'yolo_dataset/data.yaml'
os.makedirs('yolo_dataset', exist_ok=True) 

with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f" data.yaml created successfully at: {yaml_path}")


if os.path.exists(yaml_path):
    print("Verification: File exists and contains:")

else:
    print(" File not created!")

In [ ]:

%uv pip install -q ultralytics --upgrade


from IPython.display import clear_output
clear_output()
print(" Ultralytics installed/updated")

In [ ]:

import torch
from ultralytics import YOLO

def train_yolo11m_scratch():
   
    torch.cuda.empty_cache()
    print(" GPU cache cleared")
    print("="*60)
    print("YOLOv11m Training - FULL Research Paper Implementation")
    print("="*60)
    
    # Build model from scratch
    print("\n Building YOLOv11m from scratch (Random Weights)...")
    model = YOLO('yolo11m.yaml')
    
   
    print("\n Starting training with research paper methodology...")
    results = model.train(
        data='yolo_dataset/data.yaml',
        
     
        device=0,
        batch=128,            
        workers=8,            
        cache=True,        
        amp=True,             
        
        
        epochs=150,           # 150 epochs for convergence
        imgsz=1024,           # resolution is 1024*1024
        patience=50,          # Don't stop early
        warmup_epochs=3,      # PAPER: Critical for scratch training
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        
        
        optimizer='SGD',      # PAPER: Explicitly used SGD
        lr0=0.01,             # PAPER: Initial LR = 0.01 (1e-2)
        lrf=0.01,             # Final LR = 0.0001
        momentum=0.937,       # PAPER: Momentum = 0.937
        weight_decay=0.0005,  # PAPER: Weight decay = 5e-4
        cos_lr=True,          # Cosine LR decay
        
        
        augment=True,
        
        # HSV Augmentation (CRITICAL for synthetic data)
        hsv_h=0.015,          # Hue shift (minimal)
        hsv_s=0.7,            # Saturation variation (strong)
        hsv_v=0.4,            # PAPER: Brightness/Intensity variation (KEY!)
        
        # Geometric Augmentation
        degrees=0.0,          # No rotation (synthetic data is aligned)
        translate=0.1,        # Slight translation
        scale=0.5,            # Scale variation
        shear=0.0,            # No shear
        perspective=0.0,      # No perspective warp
        flipud=0.5,           # PAPER: Vertical flip
        fliplr=0.5,           # PAPER: Horizontal flip
        
        # Advanced Augmentation
        mosaic=1.0,           # CRITICAL: Mosaic for multi-scale learning
        mixup=0.0,            # Disabled (not in paper)
        copy_paste=0.0,       # Disabled
        
        # PAPER: Disable augmentation for last 20 epochs (fine-tuning)
        close_mosaic=20,      # Turn off mosaic last 20 epochs
        
      
        project='solidworks_runs',
        name='yolo11m_paper_full',
        exist_ok=True,
        plots=True,
        save=True,
        save_period=50,       # Save checkpoint every 50 epochs
        pretrained=False,     # From scratch (no pretrained weights)
        verbose=True
    )
    
    print("\n" + "="*60)
    print(" Training complete with FULL paper optimizations!")
    print("="*60)
    print(f" Best model: solidworks_runs/yolo11m_paper_full/weights/best.pt")
    print(f" Last model: solidworks_runs/yolo11m_paper_full/weights/last.pt")
    print("="*60)
    
    return results

# Execute training
if __name__ == "__main__":
    print(" Starting paper-optimized training!")
    train_yolo11m_scratch()

In [ ]:
import os
import glob
import pandas as pd
from IPython.display import Image, display

print(f"Current Working Directory: {os.getcwd()}\n")

# 1. Search for results.csv recursively to find the actual path
print(" Searching for 'results.csv' in all subdirectories...")
found_files = glob.glob('**/results.csv', recursive=True)

if not found_files:
    print(" Could not find 'results.csv' anywhere. ")
    print("   - Did the training complete?")
    print("   - If using Google Colab, is your Drive mounted?")
else:
    # Use the most likely result (usually the last one if multiple exist)
    target_file = found_files[-1] 
    run_dir = os.path.dirname(target_file)
    
    print(f" Found results at: {run_dir}")
    print("   Updating 'run_dir' and generating report...\n")
    print("="*60)

    # --- NOW RUN YOUR ORIGINAL VISUALIZATION CODE WITH THE CORRECT PATH ---
    
    csv_path = os.path.join(run_dir, "results.csv")
    
    # --- A. Display Key Graphs ---
    print("\n Key Training Graphs:\n")
    graphs = ['results.png', 'confusion_matrix.png', 'F1_curve.png', 'PR_curve.png']
    
    for graph in graphs:
        path = os.path.join(run_dir, graph)
        if os.path.exists(path):
            display(Image(filename=path))
        else:
            print(f"   (Graph {graph} not generated or not found)")

    # --- B. Metrics Summary ---
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        
        # Get best epoch
        best_idx = df['metrics/mAP50-95(B)'].idxmax()
        best_row = df.iloc[best_idx]
        
        print("\n BEST EPOCH SUMMARY")
        print("-" * 30)
        print(f"Epoch: {best_idx + 1}")
        print(f"mAP@50    : {best_row['metrics/mAP50(B)']:.4f}")
        print(f"mAP@50-95 : {best_row['metrics/mAP50-95(B)']:.4f}")
        print(f"Precision : {best_row['metrics/precision(B)']:.4f}")
        print(f"Recall    : {best_row['metrics/recall(B)']:.4f}")
        print("-" * 30)

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO
from collections import Counter

# 1. SETUP PATHS
# Update this path to where your trained model is saved!
model_path = '/root/solidworks_runs/yolo11m_paper_full/weights/best.pt' 
test_dir = '/root/solid_ai_dataset/Solid_Ai_Hackathon/test/test'
output_filename = 'submission.csv'

# 2. LOAD MODEL
print(f" Loading model from: {model_path}")
model = YOLO(model_path)

# 3. DEFINE TARGET COLUMNS
# These must match your model's class names exactly or you need a mapper.
# Assuming model class names are lowercase: 'bolt', 'locatingpin', 'nut', 'washer'
target_columns = ['bolt', 'locatingpin', 'nut', 'washer']

# 4. RUN INFERENCE
print(" Starting inference on test images...")

# stream=True is critical for large datasets to avoid running out of RAM
results_generator = model.predict(
    source=test_dir, 
    conf=0.25,      # Confidence threshold (adjust if needed)
    stream=True,    # Use generator to save memory
    verbose=False   # Keep log clean
)

data = []

for result in results_generator:
    # Get the image filename (e.g., 'image_001.jpg') from the full path
    full_path = result.path
    file_name = os.path.basename(full_path)
    
    # Get the detected class IDs
    # result.boxes.cls returns a tensor of class indices like [0, 1, 0, 3]
    class_indices = result.boxes.cls.cpu().numpy().astype(int)
    
    # Map indices to names (e.g., 0 -> 'bolt')
    detected_names = [model.names[i] for i in class_indices]
    
    # Count occurrences of each class
    counts = Counter(detected_names)
    
    # Prepare the row dictionary
    row = {'image_name': file_name}
    
    # Fill in counts for the specific target columns (default to 0 if not found)
    for col in target_columns:
        row[col] = counts.get(col, 0)
        
    data.append(row)

# 5. CREATE DATAFRAME & SAVE
df = pd.DataFrame(data)

# Ensure columns are in the correct order: image_name, bolt, locatingpin, nut, washer
df = df[['image_name'] + target_columns]

# Sort by image_name if required by competition (optional but good practice)
df = df.sort_values('image_name')

df.to_csv(output_filename, index=False)

# --- Verification ---
print(f"\n Successfully created '{output_filename}' with {len(df)} rows.")
print("\nPreview of the first 5 rows:")
print(df.head())

**Upload your model on Hugging Face**

In [ ]:
# 1. Install huggingface_hub using uv
%uv pip install huggingface_hub

import os
from huggingface_hub import HfApi, login


# Get your token from: https://huggingface.co/settings/tokens
TOKEN = ""      # <--- PASTE WRITE TOKEN HERE
REPO_ID = ""      # <--- REPLACE with your username/repo
FOLDER = "/root/solidworks_runs/yolo11m_paper_full"

# 2. Login
print(" Logging in...")
login(token=TOKEN)

# 3. Create Repo (if it doesn't exist) & Upload
api = HfApi()
print(f" Creating repo '{REPO_ID}' if missing...")
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

print(f" Uploading '{FOLDER}' to Hugging Face...")
api.upload_folder(
    folder_path=FOLDER,
    repo_id=REPO_ID,
    repo_type="model"
)

print(f" Success! View your model here: https://huggingface.co/{REPO_ID}")